# ForenSynth-X+ | Entity Resolution Pipeline
**Version:** 4.0.0

LLM-powered neuro-symbolic entity resolution for multi-modal forensic observations.

## Pipeline (12 stages)
Intake → Normalization → Blocking → Feature Computation → Scoring → Classification → Graph → Clustering → Attachment → Conflict Detection → Entity Labeling → Packaging

## v4 highlights
- **EntityCoreferenceAgent** (Groq LLM): links different aliases across modalities
- **ContextScoringAgent** (Groq LLM): semantic content similarity with batching
- Heuristic fallback when `GROQ_API_KEY` is absent
- Cross-modal clustering for timeline-ready canonical entities


## 1. Install dependencies


In [1]:
%pip install -q rapidfuzz groq networkx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 13.7 MB/s eta 0:00:00


## 2. Imports & API key setup


In [2]:
from __future__ import annotations

import hashlib
import itertools
import json
import logging
import os
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, FrozenSet, List, Optional, Set, Tuple

import networkx as nx
from rapidfuzz import fuzz

try:
    from groq import Groq as _GroqClient  # type: ignore
    _GROQ_AVAILABLE = True
except ImportError:
    _GROQ_AVAILABLE = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s – %(message)s",
)
log = logging.getLogger("forensynth_x_plus")

# Portable API key: Colab secret OR local env var
if not os.environ.get("GROQ_API_KEY"):
    try:
        from google.colab import userdata  # type: ignore
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        pass


## 3. Constants & configuration


In [3]:
ACTION_TOKENS: Set[str] = {
    "enter", "exit", "withdraw", "deposit", "call", "message",
    "send", "receive", "transfer", "arrive", "depart", "access",
    "attempt", "fail", "succeed", "heading", "headed", "inside", "clear",
}

CONFLICTING_ACTION_PAIRS: List[Tuple[str, str]] = [
    ("enter", "exit"),
    ("withdraw", "deposit"),
    ("arrive", "depart"),
    ("send", "receive"),
    ("succeed", "fail"),
    ("heading", "exited"),
]

WEIGHT_MAP: Dict[str, float] = {
    "entity_coreference":  0.280,
    "mention_consistency": 0.120,
    "temporal":            0.150,
    "location":            0.100,
    "context":             0.180,
    "lexical":             0.050,
    "interaction":         0.070,
    "modality":            0.050,
}
assert abs(sum(WEIGHT_MAP.values()) - 1.0) < 1e-9

CONFIRMED_THRESHOLD:      float = 0.80
CANDIDATE_THRESHOLD_HIGH: float = 0.65
CANDIDATE_THRESHOLD_LOW:  float = 0.50
ATTACHMENT_THRESHOLD:     float = 0.70
CLUSTER_CONFIDENCE_FLOOR: float = 0.55
CROSS_MODAL_MERGE_MIN:    float = 0.58
MERGE_COMPOSITE_MIN:      float = 0.55

OVERSIZED_CLUSTER_FACTOR: float = 3.0
TEMPORAL_WINDOW_SEC:      int   = 300
MAX_TEMPORAL_GAP_SEC:     int   = 3600
MAX_PAIRS:                int   = 500
LLM_BATCH_CHUNK_SIZE:     int   = 40

GROQ_MODEL: str = "llama-3.1-8b-instant"

FEATURE_NAMES: Tuple[str, ...] = (
    "entity_coreference", "mention_consistency", "temporal", "location",
    "context", "lexical", "interaction", "modality",
)


## 4. Data classes


In [4]:
@dataclass
class Observation:
    obs_id:     str
    entity:     str
    role:       str
    modality:   str
    location:   str
    content:    str
    timestamp:  str
    confidence: float
    entity_norm:     str   = ""
    time_offset_sec: int   = 0
    _ts_epoch:       float = field(default=0.0, repr=False)


@dataclass
class HumanConstraints:
    must_merge:     List[Tuple[str, str]]        = field(default_factory=list)
    must_not_merge: List[Tuple[str, str]]        = field(default_factory=list)
    soft_hints:     Dict[Tuple[str, str], float] = field(default_factory=dict)


@dataclass
class PairFeatures:
    obs_a: Observation
    obs_b: Observation
    entity_coreference:  float = 0.0
    mention_consistency: float = 0.0
    temporal:            float = 0.0
    location:            float = 0.0
    context:             float = 0.0
    lexical:             float = 0.0
    interaction:         float = 0.0
    modality:            float = 0.0
    composite:           float = 0.0
    reasons:             List[str] = field(default_factory=list)
    hard_negative:       bool = False

    def compute_composite(self, soft_hint: float = 0.0) -> float:
        raw = sum(getattr(self, name) * WEIGHT_MAP[name] for name in FEATURE_NAMES)
        self.composite = min(1.0, raw + soft_hint)
        return self.composite


@dataclass
class EdgeRecord:
    alias_1:        str
    alias_2:        str
    obs_id_1:       str
    obs_id_2:       str
    weight:         float
    support:        int = 1
    classification: str = "rejected"
    features:       Optional[PairFeatures] = None
    reasons:        List[str] = field(default_factory=list)
    hard_negative:  bool = False


## 5. LLM batch helpers


In [5]:
def _normalize_alias(alias: str) -> str:
    return re.sub(r"[^a-z0-9_]", "_", alias.strip().lower())


def _parse_llm_json_map(text: str) -> Dict[str, float]:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"```\s*$", "", text, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{[^{}]*\}", text, flags=re.DOTALL)
        if not match:
            raise
        parsed = json.loads(match.group(0))
    return {str(k): float(v) for k, v in parsed.items() if re.match(r"^\d+$", str(k))}


def _mention_payload(obs: Observation) -> Dict[str, str]:
    return {
        "entity":    obs.entity,
        "role":      obs.role,
        "modality":  obs.modality,
        "location":  obs.location[:120],
        "content":   obs.content[:220],
        "timestamp": obs.timestamp,
    }


class _BatchLLMScorer:
    """Shared Groq batch caller with chunking and local fallback."""

    def __init__(self, model: str, enabled: bool):
        self.model = model
        self.enabled = enabled and _GROQ_AVAILABLE
        self._client: Any = None
        if self.enabled:
            api_key = os.environ.get("GROQ_API_KEY", "")
            if api_key:
                self._client = _GroqClient(api_key=api_key)
            else:
                log.warning("GROQ_API_KEY not set – using heuristic/fuzzy fallbacks.")
                self.enabled = False

    def score_batch(
        self,
        payload_items: List[Dict[str, Any]],
        system_prompt: str,
        user_intro: str,
        fallback_fn,
    ) -> List[float]:
        """Return one score per payload item (index-aligned)."""
        n = len(payload_items)
        if n == 0:
            return []

        scores: List[Optional[float]] = [None] * n
        if not self.enabled or self._client is None:
            return [float(fallback_fn(i)) for i in range(n)]

        for chunk_start in range(0, n, LLM_BATCH_CHUNK_SIZE):
            chunk = payload_items[chunk_start : chunk_start + LLM_BATCH_CHUNK_SIZE]
            prompt = (
                f"{system_prompt}\n\n{user_intro}\n"
                + json.dumps(chunk, ensure_ascii=False)
                + '\n\nReturn ONLY JSON: {"0": 0.85, "1": 0.12, ...} with one float 0.0-1.0 per id.'
            )
            max_tok = min(4096, max(256, len(chunk) * 16 + 128))
            raw_scores: Dict[str, float] = {}
            try:
                resp = self._client.chat.completions.create(
                    model=self.model,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=max_tok,
                    temperature=0.0,
                )
                raw_scores = _parse_llm_json_map(resp.choices[0].message.content or "")
            except Exception as exc:
                log.warning("LLM batch chunk failed (%s); using fallback for %d pairs.", exc, len(chunk))

            for local_i in range(len(chunk)):
                global_i = chunk_start + local_i
                if str(local_i) in raw_scores:
                    scores[global_i] = max(0.0, min(1.0, raw_scores[str(local_i)]))
                else:
                    scores[global_i] = float(fallback_fn(global_i))

        return [s if s is not None else float(fallback_fn(i)) for i, s in enumerate(scores)]


## 6. Context scoring agent (LLM)


In [6]:
class ContextScoringAgent:
  def __init__(self, model: str = GROQ_MODEL, enabled: bool = True):
    self._llm = _BatchLLMScorer(model, enabled)
    self._cache: Dict[Tuple[str, str], float] = {}

  @staticmethod
  def _cache_key(a: str, b: str) -> Tuple[str, str]:
    ka, kb = a[:120], b[:120]
    return (ka, kb) if ka <= kb else (kb, ka)

  def _detect_conflicting_actions(self, text_a: str, text_b: str) -> float:
    def _tokens(t: str) -> Set[str]:
      return ACTION_TOKENS & set(re.findall(r"\b\w+\b", t.lower()))
    ta, tb = _tokens(text_a), _tokens(text_b)
    for act_a, act_b in CONFLICTING_ACTION_PAIRS:
      if (act_a in ta and act_b in tb) or (act_b in ta and act_a in tb):
        return 0.5
    return 1.0

  def _content_fallback(self, pair: Tuple[str, str]) -> float:
    ca, cb = pair
    penalty = self._detect_conflicting_actions(ca, cb)
    return min(1.0, max(0.0, fuzz.token_set_ratio(ca, cb) / 100.0 * penalty))

  def precompute_batch_scores(
    self,
    candidate_pairs: List[Tuple[str, str]],
    obs_by_id: Dict[str, Observation],
  ) -> None:
    unique: List[Tuple[str, str]] = []
    seen: Set[Tuple[str, str]] = set()
    for id_a, id_b in candidate_pairs:
      oa, ob = obs_by_id.get(id_a), obs_by_id.get(id_b)
      if not oa or not ob:
        continue
      ck = self._cache_key(oa.content, ob.content)
      if ck in self._cache or ck in seen:
        continue
      seen.add(ck)
      unique.append(ck)

    if not unique:
      return

    log.info("Context precompute: %d unique content pairs.", len(unique))
    system = (
      "You are a forensic semantic analyst. Score how likely two observation texts "
      "describe the same underlying event or the same actor's actions."
    )
    intro = "Rate each text pair from 0.0 (unrelated) to 1.0 (same event/entity)."
    payload = [
      {"id": str(i), "text_a": a[:300], "text_b": b[:300]}
      for i, (a, b) in enumerate(unique)
    ]

    def fallback(i: int) -> float:
      a, b = unique[i]
      return self._content_fallback((a, b))

    scored = self._llm.score_batch(payload, system, intro, fallback)
    for pair_key, raw, (a, b) in zip(unique, scored, unique):
      penalty = self._detect_conflicting_actions(a, b)
      self._cache[pair_key] = min(1.0, max(0.0, raw * penalty))

  def score(self, content_a: str, content_b: str) -> float:
    ck = self._cache_key(content_a, content_b)
    if ck not in self._cache:
      self._cache[ck] = self._content_fallback(ck)
    return self._cache[ck]


## 7. Entity co-reference agent (LLM)


In [7]:
class EntityCoreferenceAgent:
  """LLM scores whether two forensic mentions refer to the same real-world entity."""

  def __init__(self, model: str = GROQ_MODEL, enabled: bool = True):
    self._llm = _BatchLLMScorer(model, enabled)
    self._cache: Dict[Tuple[str, str], float] = {}

  @staticmethod
  def _pair_key(id_a: str, id_b: str) -> Tuple[str, str]:
    return (id_a, id_b) if id_a <= id_b else (id_b, id_a)

  @staticmethod
  def heuristic_coreference(oa: Observation, ob: Observation, temporal_window: int, max_gap: int) -> float:
    if oa.entity_norm == ob.entity_norm:
      return 1.0
    if oa.role.strip().lower() != ob.role.strip().lower():
      return 0.12

    dt = abs(oa.time_offset_sec - ob.time_offset_sec)
    if dt > max_gap:
      return 0.05

    cross_modal = oa.modality.lower() != ob.modality.lower()
    score = 0.45 if cross_modal else 0.32

    if dt <= temporal_window:
      score += 0.25
    elif dt <= max_gap:
      score += 0.10

    la, lb = oa.location.strip().lower(), ob.location.strip().lower()
    if la and lb:
      loc_sim = fuzz.token_set_ratio(la, lb) / 100.0
      if loc_sim >= 0.45:
        score += 0.10
      shared_tokens = set(re.findall(r"[a-z0-9]+", la)) & set(re.findall(r"[a-z0-9]+", lb))
      if shared_tokens & {"atm", "server", "booth", "room", "network", "ssh", "login"}:
        score += 0.08

    content_sim = fuzz.token_set_ratio(oa.content, ob.content) / 100.0
    if content_sim >= 0.35:
      score += 0.08

    topical = {"enter", "entered", "inside", "heading", "headed", "starting", "exit", "exited",
               "coming", "leaving", "depart", "server", "ssh", "login", "access", "atm", "booth", "clear"}
    ca = set(re.findall(r"[a-z0-9]+", oa.content.lower()))
    cb = set(re.findall(r"[a-z0-9]+", ob.content.lower()))
    overlap = ca & cb & topical
    if overlap:
      score += min(0.14, 0.05 * len(overlap))

    enter_tokens = {"enter", "entered", "inside", "heading", "headed", "starting"}
    exit_tokens = {"exit", "exited", "coming", "leaving", "depart"}
    a_enter = bool(ca & enter_tokens)
    b_enter = bool(cb & enter_tokens)
    a_exit = bool(ca & exit_tokens)
    b_exit = bool(cb & exit_tokens)
    if (a_enter and b_enter) or (a_exit and b_exit):
      score += 0.10

    # Narrative phase bucketing: approach / inside / exit within one event window
    def _phase(tokens: Set[str]) -> str:
      if tokens & exit_tokens:
        return "exit"
      if tokens & enter_tokens:
        return "inside"
      if tokens & {"heading", "headed", "towards", "see", "activity"}:
        return "approach"
      return "other"

    if dt <= temporal_window and _phase(ca) == _phase(cb) != "other":
      score += 0.08

    return min(1.0, score)

  def precompute_batch_scores(
    self,
    candidate_pairs: List[Tuple[str, str]],
    obs_by_id: Dict[str, Observation],
    temporal_window: int,
    max_gap: int,
  ) -> None:
    unique_keys: List[Tuple[str, str]] = []
    seen: Set[Tuple[str, str]] = set()
    for id_a, id_b in candidate_pairs:
      key = self._pair_key(id_a, id_b)
      if key in self._cache or key in seen:
        continue
      seen.add(key)
      unique_keys.append(key)

    if not unique_keys:
      return

    log.info("Entity coreference precompute: %d unique mention pairs.", len(unique_keys))
    obs_pairs = [(obs_by_id[a], obs_by_id[b]) for a, b in unique_keys]
    payload = [
      {"id": str(i), "mention_a": _mention_payload(oa), "mention_b": _mention_payload(ob)}
      for i, (oa, ob) in enumerate(obs_pairs)
    ]

    system = (
      "You are a digital forensics entity-resolution expert. Decide if mention A and "
      "mention B refer to the SAME real-world person/device/account, even when names differ "
      "(e.g. 'Suspect A', an IP, and 'person in red jacket'). Use role, modality, location, "
      "timestamp, and content together. Different roles (suspect vs witness) should score low."
    )
    intro = "Score each mention pair from 0.0 (definitely different) to 1.0 (same entity)."

    def fallback(i: int) -> float:
      oa, ob = obs_pairs[i]
      return self.heuristic_coreference(oa, ob, temporal_window, max_gap)

    scored = self._llm.score_batch(payload, system, intro, fallback)
    for key, raw in zip(unique_keys, scored):
      self._cache[key] = max(0.0, min(1.0, raw))

  def score(self, oa: Observation, ob: Observation, temporal_window: int, max_gap: int) -> float:
    key = self._pair_key(oa.obs_id, ob.obs_id)
    if key not in self._cache:
      self._cache[key] = self.heuristic_coreference(oa, ob, temporal_window, max_gap)
    return self._cache[key]


## 8. Union-Find clustering


In [8]:
class UnionFind:
  def __init__(self, elements: List[str]):
    self.parent = {e: e for e in elements}
    self.rank = {e: 0 for e in elements}

  def find(self, x: str) -> str:
    while self.parent[x] != x:
      self.parent[x] = self.parent[self.parent[x]]
      x = self.parent[x]
    return x

  def union(self, x: str, y: str) -> bool:
    px, py = self.find(x), self.find(y)
    if px == py:
      return False
    if self.rank[px] < self.rank[py]:
      px, py = py, px
    self.parent[py] = px
    if self.rank[px] == self.rank[py]:
      self.rank[px] += 1
    return True

  def groups(self) -> Dict[str, List[str]]:
    d: Dict[str, List[str]] = defaultdict(list)
    for e in self.parent:
      d[self.find(e)].append(e)
    return dict(d)


## 9. Entity resolution pipeline (12 stages)


In [9]:
class EntityResolutionPipeline:
  def __init__(
    self,
    config: Optional[Dict[str, Any]] = None,
    human_constraints: Optional[HumanConstraints] = None,
    llm_enabled: bool = True,
  ):
    cfg = config or {}
    self.check_duplicates = cfg.get("check_duplicates", True)
    self.case_base_time = cfg.get("case_base_time", None)
    self.temporal_window_sec = cfg.get("temporal_window_sec", TEMPORAL_WINDOW_SEC)
    self.max_temporal_gap_sec = cfg.get("max_temporal_gap_sec", MAX_TEMPORAL_GAP_SEC)
    self.max_pairs = cfg.get("max_pairs", MAX_PAIRS)
    self.confirmed_threshold = cfg.get("confirmed_threshold", CONFIRMED_THRESHOLD)
    self.candidate_threshold_low = cfg.get("candidate_threshold_low", CANDIDATE_THRESHOLD_LOW)
    self.candidate_threshold_high = cfg.get("candidate_threshold_high", CANDIDATE_THRESHOLD_HIGH)
    self.llm_enabled = llm_enabled

    self.constraints = human_constraints or HumanConstraints()
    self.context_agent = ContextScoringAgent(model=GROQ_MODEL, enabled=self.llm_enabled)
    self.entity_agent = EntityCoreferenceAgent(model=GROQ_MODEL, enabled=self.llm_enabled)

    self.case_id = "UNKNOWN_CASE"
    self.observations: List[Observation] = []
    self.base_epoch = 0.0
    self.candidate_pairs: List[Tuple[str, str]] = []
    self.edges: List[EdgeRecord] = []
    self.graph = nx.Graph()
    self.clusters: Dict[str, List[str]] = {}
    self.conflicts: List[Dict[str, Any]] = []
    self.status = "success"
    self.error_message = ""
    self.stage_timings: Dict[str, float] = {}
    self._obs_by_id: Dict[str, Observation] = {}
    self._pair_features: Dict[Tuple[str, str], PairFeatures] = {}
    self._candidate_data: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    self._canonical_entities: List[Dict[str, Any]] = []

  def run(self, raw_input: Dict[str, Any]) -> Dict[str, Any]:
    pipeline_start = time.perf_counter()
    try:
      self._stage(1, self._stage_1_intake, raw_input)
      self._stage(2, self._stage_2_normalization)
      self._stage(3, self._stage_3_blocking)
      self._stage(4, self._stage_4_feature_computation)
      self._stage(5, self._stage_5_scoring)
      self._stage(6, self._stage_6_classification)
      self._stage(7, self._stage_7_graph_construction)
      self._stage(8, self._stage_8_clustering)
      self._stage(9, self._stage_9_guarded_attachment)
      self._stage(10, self._stage_10_conflict_detection)
      self._stage(11, self._stage_11_entity_labeling)
    except Exception as exc:
      log.exception("Pipeline failed: %s", exc)
      self.status = "failed"
      self.error_message = str(exc)
    return self._stage_12_package(time.perf_counter() - pipeline_start)

  def _stage(self, n: int, fn: Any, *args: Any) -> None:
    t0 = time.perf_counter()
    fn(*args)
    suffix = fn.__name__.split("_stage_")[1]
    self.stage_timings[f"stage_{suffix}"] = time.perf_counter() - t0
    log.info("Stage %d (%s) done in %.4fs", n, fn.__name__, self.stage_timings[f"stage_{suffix}"])

  @staticmethod
  def _canonical_pair(a: str, b: str) -> Tuple[str, str]:
    return (min(a, b), max(a, b))

  @staticmethod
  def _parse_ts(ts_str: str) -> float:
    for fmt in ("%Y-%m-%dT%H:%M:%S", "%Y-%m-%dT%H:%M:%SZ", "%Y-%m-%dT%H:%M:%S%z", "%Y-%m-%d %H:%M:%S"):
      try:
        dt = datetime.strptime(ts_str.strip(), fmt)
        if dt.tzinfo is None:
          dt = dt.replace(tzinfo=timezone.utc)
        return dt.timestamp()
      except ValueError:
        continue
    return 0.0

  def _stage_1_intake(self, raw_input: Dict[str, Any]) -> None:
    self.case_id = str(raw_input.get("case_id", "") or "UNKNOWN_CASE")
    raw_obs = raw_input.get("observations", [])
    seen_hashes: Dict[str, Observation] = {}
    for item in raw_obs:
      try:
        obs = Observation(
          obs_id=str(item["obs_id"]),
          entity=str(item["entity"]),
          role=str(item.get("role", "unknown")),
          modality=str(item.get("modality", "unknown")),
          location=str(item.get("location", "")),
          content=str(item.get("content", "")),
          timestamp=str(item.get("timestamp", "")),
          confidence=float(item.get("confidence", 0.5)),
        )
      except (KeyError, TypeError, ValueError) as exc:
        log.warning("Skipping malformed obs %s: %s", item.get("obs_id"), exc)
        continue
      obs.confidence = max(0.0, min(1.0, obs.confidence))
      if self.check_duplicates:
        fp = f"{obs.entity}|{obs.modality}|{obs.location}|{obs.content}|{obs.timestamp}"
        digest = hashlib.sha256(fp.encode()).hexdigest()
        if digest not in seen_hashes or obs.confidence > seen_hashes[digest].confidence:
          seen_hashes[digest] = obs
      else:
        seen_hashes[obs.obs_id] = obs
    self.observations = list(seen_hashes.values())
    self._obs_by_id = {o.obs_id: o for o in self.observations}

  def _stage_2_normalization(self) -> None:
    epochs = [self._parse_ts(o.timestamp) for o in self.observations]
    valid = [e for e in epochs if e > 0]
    self.base_epoch = min(valid) if valid else 0.0
    for obs, epoch in zip(self.observations, epochs):
      obs._ts_epoch = epoch
      obs.time_offset_sec = int(epoch - self.base_epoch) if epoch > 0 else 0
      obs.entity_norm = _normalize_alias(obs.entity)

  def _build_obs_blocked_set(self) -> Set[Tuple[str, str]]:
    blocked: Set[Tuple[str, str]] = set()
    for a_alias, b_alias in self.constraints.must_not_merge:
      a_norm, b_norm = _normalize_alias(a_alias), _normalize_alias(b_alias)
      for oa in self.observations:
        for ob in self.observations:
          if oa.obs_id == ob.obs_id:
            continue
          if {oa.entity_norm, ob.entity_norm} == {a_norm, b_norm}:
            blocked.add(self._canonical_pair(oa.obs_id, ob.obs_id))
    return blocked

  def _stage_3_blocking(self) -> None:
    blocked = self._build_obs_blocked_set()
    forced: Set[Tuple[str, str]] = set()
    for a_alias, b_alias in self.constraints.must_merge:
      a_norm, b_norm = _normalize_alias(a_alias), _normalize_alias(b_alias)
      ids_a = [o.obs_id for o in self.observations if o.entity_norm == a_norm]
      ids_b = [o.obs_id for o in self.observations if o.entity_norm == b_norm]
      for ia, ib in itertools.product(ids_a, ids_b):
        if ia != ib:
          key = self._canonical_pair(ia, ib)
          if key not in blocked:
            forced.add(key)

    candidate_set: Set[Tuple[str, str]] = set()
    obs_list = self.observations
    for i, oa in enumerate(obs_list):
      for ob in obs_list[i + 1 :]:
        key = self._canonical_pair(oa.obs_id, ob.obs_id)
        if key in blocked:
          continue
        dt = abs(oa.time_offset_sec - ob.time_offset_sec)
        same_loc = bool(oa.location.strip()) and oa.location.strip().lower() == ob.location.strip().lower()
        same_role = oa.role.strip().lower() == ob.role.strip().lower()
        cross_modal = oa.modality.lower() != ob.modality.lower()
        if same_loc or dt <= self.temporal_window_sec or (cross_modal and same_role and dt <= self.max_temporal_gap_sec):
          candidate_set.add(key)

    all_candidates = list(forced) + [p for p in candidate_set if p not in forced]
    self.candidate_pairs = all_candidates[: self.max_pairs]

  def _feat_mention_consistency(self, oa: Observation, ob: Observation) -> Tuple[float, bool]:
    if oa.modality == ob.modality:
      if oa.entity_norm == ob.entity_norm:
        dt = abs(oa.time_offset_sec - ob.time_offset_sec)
        loc = oa.location.strip().lower() == ob.location.strip().lower() if oa.location and ob.location else False
        if dt <= self.temporal_window_sec and loc:
          return 0.85, True
        if dt <= self.max_temporal_gap_sec:
          return 0.45, False
        return 0.20, False
      return 0.30, False
    dt = abs(oa.time_offset_sec - ob.time_offset_sec)
    if dt <= self.temporal_window_sec:
      return 0.78, True
    if dt <= self.max_temporal_gap_sec:
      return 0.42, False
    return 0.10, False

  def _feat_temporal(self, oa: Observation, ob: Observation) -> float:
    dt = abs(oa.time_offset_sec - ob.time_offset_sec)
    if dt > self.max_temporal_gap_sec:
      return 0.0
    if dt <= self.temporal_window_sec:
      return max(0.0, 1.0 - dt / self.temporal_window_sec)
    span = self.max_temporal_gap_sec - self.temporal_window_sec
    return max(0.0, 0.3 * (1.0 - (dt - self.temporal_window_sec) / span))

  def _feat_location(self, oa: Observation, ob: Observation) -> float:
    la, lb = oa.location.strip().lower(), ob.location.strip().lower()
    if not la or not lb:
      return 0.35
    if la == lb:
      return 1.0
    return fuzz.token_set_ratio(la, lb) / 100.0

  def _feat_context(self, oa: Observation, ob: Observation) -> float:
    return self.context_agent.score(oa.content, ob.content)

  def _feat_lexical(self, oa: Observation, ob: Observation) -> float:
    return fuzz.token_sort_ratio(oa.entity_norm, ob.entity_norm) / 100.0

  def _feat_interaction(self, oa: Observation, ob: Observation) -> float:
    modality_pairs = {
      frozenset({"video", "audio"}): 0.85,
      frozenset({"video", "text"}): 0.72,
      frozenset({"audio", "text"}): 0.68,
      frozenset({"video"}): 0.50,
      frozenset({"audio"}): 0.50,
      frozenset({"text"}): 0.45,
    }
    base = modality_pairs.get(frozenset({oa.modality.lower(), ob.modality.lower()}), 0.30)
    if oa.role and ob.role and oa.role.lower() != ob.role.lower():
      conflicting = {frozenset({"suspect", "witness"}), frozenset({"suspect", "victim"}), frozenset({"perpetrator", "victim"})}
      if frozenset({oa.role.lower(), ob.role.lower()}) in conflicting:
        base *= 0.55
    return min(1.0, base)

  def _feat_modality(self, oa: Observation, ob: Observation) -> float:
    return 0.40 if oa.modality.lower() == ob.modality.lower() else 0.82

  def _feat_entity_coreference(self, oa: Observation, ob: Observation) -> float:
    return self.entity_agent.score(oa, ob, self.temporal_window_sec, self.max_temporal_gap_sec)

  def _stage_4_feature_computation(self) -> None:
    self.context_agent.precompute_batch_scores(self.candidate_pairs, self._obs_by_id)
    self.entity_agent.precompute_batch_scores(
      self.candidate_pairs, self._obs_by_id, self.temporal_window_sec, self.max_temporal_gap_sec
    )
    self._pair_features = {}
    for key in self.candidate_pairs:
      oa, ob = self._obs_by_id.get(key[0]), self._obs_by_id.get(key[1])
      if not oa or not ob:
        continue
      pf = PairFeatures(obs_a=oa, obs_b=ob)
      pf.entity_coreference = self._feat_entity_coreference(oa, ob)
      pf.mention_consistency, _ = self._feat_mention_consistency(oa, ob)
      pf.temporal = self._feat_temporal(oa, ob)
      pf.location = self._feat_location(oa, ob)
      pf.context = self._feat_context(oa, ob)
      pf.lexical = self._feat_lexical(oa, ob)
      pf.interaction = self._feat_interaction(oa, ob)
      pf.modality = self._feat_modality(oa, ob)
      self._pair_features[key] = pf

  def _stage_5_scoring(self) -> None:
    blocked_alias_pairs = {
      self._canonical_pair(_normalize_alias(a), _normalize_alias(b))
      for a, b in self.constraints.must_not_merge
    }
    for key, pf in self._pair_features.items():
      oa, ob = pf.obs_a, pf.obs_b
      hint_key = self._canonical_pair(oa.entity_norm, ob.entity_norm)
      pf.compute_composite(soft_hint=self.constraints.soft_hints.get(hint_key, 0.0))
      pf.reasons = [name for name in FEATURE_NAMES if getattr(pf, name) > 0.5]
      if self._canonical_pair(oa.entity_norm, ob.entity_norm) in blocked_alias_pairs:
        pf.composite = 0.0
        pf.reasons = []
        pf.hard_negative = True

  def _stage_6_classification(self) -> None:
    self.edges = []
    for pf in self._pair_features.values():
      if pf.hard_negative:
        cls = "rejected"
      elif pf.composite >= self.confirmed_threshold:
        cls = "confirmed"
      elif pf.composite >= self.candidate_threshold_high:
        cls = "likely"
      elif pf.composite >= self.candidate_threshold_low:
        cls = "possible"
      else:
        cls = "rejected"
      self.edges.append(
        EdgeRecord(
          alias_1=pf.obs_a.entity_norm,
          alias_2=pf.obs_b.entity_norm,
          obs_id_1=pf.obs_a.obs_id,
          obs_id_2=pf.obs_b.obs_id,
          weight=pf.composite,
          classification=cls,
          features=pf,
          reasons=list(pf.reasons),
          hard_negative=pf.hard_negative,
        )
      )

  def _stage_7_graph_construction(self) -> None:
    G = nx.Graph()
    for obs in self.observations:
      G.add_node(obs.obs_id)
    for edge in self.edges:
      if edge.classification != "confirmed":
        continue
      if G.has_edge(edge.obs_id_1, edge.obs_id_2):
        G[edge.obs_id_1][edge.obs_id_2]["weight"] = max(G[edge.obs_id_1][edge.obs_id_2]["weight"], edge.weight)
        G[edge.obs_id_1][edge.obs_id_2]["support"] += 1
      else:
        G.add_edge(edge.obs_id_1, edge.obs_id_2, weight=edge.weight, support=1)
    self.graph = G

  def _stage_8_clustering(self) -> None:
    components = {f"C{i+1}": sorted(comp) for i, comp in enumerate(nx.connected_components(self.graph))}
    uf = UnionFind([o.obs_id for o in self.observations])
    for comp in components.values():
      for j in range(1, len(comp)):
        uf.union(comp[0], comp[j])

    for edge in self.edges:
      if edge.hard_negative or edge.classification == "rejected":
        continue
      oa, ob = self._obs_by_id[edge.obs_id_1], self._obs_by_id[edge.obs_id_2]
      dt = abs(oa.time_offset_sec - ob.time_offset_sec)
      temporal_ok = dt <= self.max_temporal_gap_sec
      role_ok = oa.role.strip().lower() == ob.role.strip().lower()
      entity_corr = edge.features.entity_coreference if edge.features else 0.0
      composite = edge.weight

      if oa.entity_norm == ob.entity_norm:
        # Same alias can move across locations during an incident.
        if temporal_ok and composite >= MERGE_COMPOSITE_MIN:
          uf.union(edge.obs_id_1, edge.obs_id_2)
      elif (
        role_ok
        and temporal_ok
        and entity_corr >= CROSS_MODAL_MERGE_MIN
        and composite >= MERGE_COMPOSITE_MIN
      ):
        uf.union(edge.obs_id_1, edge.obs_id_2)

    self.clusters = {f"C{i+1}": sorted(m) for i, (_, m) in enumerate(uf.groups().items())}

  def _stage_9_guarded_attachment(self) -> None:
    obs_to_cluster = {oid: cid for cid, members in self.clusters.items() for oid in members}
    self._candidate_data = defaultdict(list)

    def is_singleton(obs_id: str) -> bool:
      cid = obs_to_cluster.get(obs_id)
      return cid is not None and len(self.clusters.get(cid, [])) == 1

    for edge in self.edges:
      if edge.classification not in ("likely", "possible") or edge.hard_negative:
        continue
      for singleton_id, partner_id in (
        (edge.obs_id_1, edge.obs_id_2) if is_singleton(edge.obs_id_1) else (None, None),
        (edge.obs_id_2, edge.obs_id_1) if is_singleton(edge.obs_id_2) else (None, None),
      ):
        if not singleton_id:
          continue
        partner_cid = obs_to_cluster.get(partner_id)
        if not partner_cid:
          continue
        if edge.weight >= ATTACHMENT_THRESHOLD:
          old_cid = obs_to_cluster[singleton_id]
          self.clusters[partner_cid].append(singleton_id)
          self.clusters[old_cid].remove(singleton_id)
          if not self.clusters[old_cid]:
            del self.clusters[old_cid]
          obs_to_cluster[singleton_id] = partner_cid
        else:
          s_obs, p_obs = self._obs_by_id[singleton_id], self._obs_by_id[partner_id]
          reasons = list(edge.reasons)
          self._candidate_data[singleton_id].append({"candidate_alias": p_obs.entity_norm, "score": round(edge.weight, 4), "reasons": reasons})
          self._candidate_data[partner_id].append({"candidate_alias": s_obs.entity_norm, "score": round(edge.weight, 4), "reasons": reasons})
    self.clusters = {k: v for k, v in self.clusters.items() if v}

  def _cluster_confidence(self, cluster_obs_ids: List[str]) -> float:
    if len(cluster_obs_ids) <= 1:
      obs = self._obs_by_id.get(cluster_obs_ids[0]) if cluster_obs_ids else None
      return obs.confidence if obs else 0.5
    member_set = set(cluster_obs_ids)
    total_weight = weighted_sum = 0.0
    for edge in self.edges:
      if edge.classification == "confirmed" and edge.obs_id_1 in member_set and edge.obs_id_2 in member_set:
        oa, ob = self._obs_by_id[edge.obs_id_1], self._obs_by_id[edge.obs_id_2]
        w = (oa.confidence + ob.confidence) / 2.0
        total_weight += w
        weighted_sum += w * edge.weight
    if total_weight == 0.0:
      confs = [self._obs_by_id[oid].confidence for oid in cluster_obs_ids if oid in self._obs_by_id]
      return sum(confs) / len(confs) if confs else 0.5
    return weighted_sum / total_weight

  def _stage_10_conflict_detection(self) -> None:
    sizes = [len(v) for v in self.clusters.values()]
    if len(sizes) >= 2:
      mean_sz = sum(sizes) / len(sizes)
      std_sz = (sum((s - mean_sz) ** 2 for s in sizes) / len(sizes)) ** 0.5
    else:
      mean_sz = sizes[0] if sizes else 1
      std_sz = 1.0
    oversized_threshold = mean_sz + OVERSIZED_CLUSTER_FACTOR * std_sz

    for cid, members in self.clusters.items():
      obs_list = [self._obs_by_id[oid] for oid in members if oid in self._obs_by_id]
      loc_time_map: Dict[Tuple[str, int], Set[str]] = defaultdict(set)
      for obs in obs_list:
        if obs.location.strip():
          loc_time_map[(obs.location.strip().lower(), obs.time_offset_sec)].add(obs.entity_norm)
      for (loc, ts), _ in loc_time_map.items():
        if any(o.time_offset_sec == ts and o.location.strip() and o.location.strip().lower() != loc for o in obs_list):
          self.conflicts.append({
            "type": "physical_impossibility",
            "cluster_id": cid,
            "detail": f"Concurrent different locations at t={ts}s in {cid}.",
          })
          self.status = "awaiting_human_validation"
          break
      cc = self._cluster_confidence(members)
      if cc < CLUSTER_CONFIDENCE_FLOOR:
        self.conflicts.append({
          "type": "low_confidence",
          "cluster_id": cid,
          "cluster_confidence": round(cc, 4),
          "detail": f"Cluster {cid} confidence {cc:.3f} below floor.",
        })
        self.status = "awaiting_human_validation"
      if oversized_threshold > 0 and len(members) > oversized_threshold:
        self.conflicts.append({
          "type": "oversized_cluster",
          "cluster_id": cid,
          "size": len(members),
          "threshold": round(oversized_threshold, 1),
          "detail": f"Cluster {cid} oversized ({len(members)}).",
        })
        self.status = "awaiting_human_validation"

  def _stage_11_entity_labeling(self) -> None:
    self._canonical_entities = []
    assigned: Set[str] = set()
    deduped: Dict[str, List[str]] = {}
    for cid, members in sorted(self.clusters.items()):
      unique = [m for m in members if m not in assigned]
      if unique:
        deduped[cid] = unique
        assigned |= set(unique)
    self.clusters = deduped

    def fmt_ts(epoch: float) -> str:
      return "" if epoch <= 0 else datetime.fromtimestamp(epoch, tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    for entity_idx, (_, members) in enumerate(self.clusters.items(), start=1):
      obs_list = [self._obs_by_id[oid] for oid in members if oid in self._obs_by_id]
      if not obs_list:
        continue
      raw_aliases = [o.entity for o in obs_list]
      primary_alias = Counter(raw_aliases).most_common(1)[0][0]
      aliases_unique = list(dict.fromkeys(raw_aliases))
      epochs = [o._ts_epoch for o in obs_list if o._ts_epoch > 0]
      member_set = set(members)
      confirmed_obs_ids: Set[str] = set()
      for e in self.edges:
        if e.classification == "confirmed" and (e.obs_id_1 in member_set or e.obs_id_2 in member_set):
          if e.obs_id_1 in member_set:
            confirmed_obs_ids.add(e.obs_id_1)
          if e.obs_id_2 in member_set:
            confirmed_obs_ids.add(e.obs_id_2)
      candidate_mentions: List[Dict[str, Any]] = []
      seen: Set[Tuple[str, str]] = set()
      for oid in members:
        for cdata in self._candidate_data.get(oid, []):
          key = (cdata["candidate_alias"], str(round(cdata["score"], 2)))
          if key not in seen:
            candidate_mentions.append({
              "candidate_alias": cdata["candidate_alias"],
              "score": cdata["score"],
              "reasons": list(cdata.get("reasons", [])),
            })
            seen.add(key)
      self._canonical_entities.append({
        "entity_id": f"entity_{entity_idx}",
        "aliases": aliases_unique,
        "primary_alias": primary_alias,
        "total_mentions": len(obs_list),
        "confirmed_mentions": sorted(confirmed_obs_ids),
        "candidate_mentions": candidate_mentions,
        "confidence_score": round(self._cluster_confidence(members), 4),
        "confirmed_edges": sum(1 for e in self.edges if e.classification == "confirmed" and (e.obs_id_1 in member_set or e.obs_id_2 in member_set)),
        "candidate_edges": sum(1 for e in self.edges if e.classification in ("likely", "possible") and not e.hard_negative and (e.obs_id_1 in member_set or e.obs_id_2 in member_set)),
        "modalities": sorted({o.modality for o in obs_list}),
        "locations": sorted({o.location for o in obs_list if o.location}),
        "roles": sorted({o.role for o in obs_list}),
        "sources": sorted({o.obs_id for o in obs_list}),
        "earliest_timestamp": fmt_ts(min(epochs) if epochs else 0),
        "latest_timestamp": fmt_ts(max(epochs) if epochs else 0),
        "time_span_seconds": int(max(epochs) - min(epochs)) if epochs else 0,
      })

  def _stage_12_package(self, total_time: float) -> Dict[str, Any]:
    cluster_output = []
    for cid, members in self.clusters.items():
      member_set = set(members)
      edge_dicts = []
      for e in self.edges:
        if e.obs_id_1 in member_set or e.obs_id_2 in member_set:
          edge_reasons = [] if e.hard_negative or e.classification == "rejected" else list(e.reasons)
          edge_dicts.append({
            "alias_1": e.alias_1,
            "alias_2": e.alias_2,
            "weight": round(e.weight, 4),
            "support": e.support,
            "classifications": {
              "confirmed": 1 if e.classification == "confirmed" else 0,
              "candidate": 1 if e.classification in ("likely", "possible") else 0,
              "rejected": 1 if e.classification == "rejected" else 0,
            },
            "hard_negative": e.hard_negative,
            "reasons": edge_reasons,
          })
      cluster_output.append({
        "cluster_id": cid,
        "size": len(members),
        "aliases": list({self._obs_by_id[oid].entity for oid in members if oid in self._obs_by_id}),
        "obs_ids": sorted(members),
        "edges": edge_dicts,
      })

    remap = {
      "stage_1_intake": "stage_1_intake",
      "stage_2_normalization": "stage_2_normalization",
      "stage_3_blocking": "stage_3_blocking",
      "stage_4_feature_computation": "stage_4_features",
      "stage_5_scoring": "stage_5_scoring",
      "stage_6_classification": "stage_6_classification",
      "stage_7_graph_construction": "stage_7_graph_building",
      "stage_8_clustering": "stage_8_clustering",
      "stage_9_guarded_attachment": "stage_9_attachment",
      "stage_10_conflict_detection": "stage_10_conflict_detection",
      "stage_11_entity_labeling": "stage_11_labeling",
    }
    final_timings = {remap.get(k, k): round(v, 6) for k, v in self.stage_timings.items()}
    final_timings["stage_12_packaging"] = 0.0

    return {
      "case_id": self.case_id,
      "status": self.status,
      "error_message": self.error_message,
      "entity_count": len(self._canonical_entities),
      "canonical_entities": self._canonical_entities,
      "clusters": cluster_output,
      "conflicts_detected": len(self.conflicts),
      "configuration": {
        "check_duplicates": self.check_duplicates,
        "case_base_time": self.case_base_time,
        "temporal_window_sec": self.temporal_window_sec,
        "max_temporal_gap_sec": self.max_temporal_gap_sec,
        "max_pairs": self.max_pairs,
        "confirmed_threshold": self.confirmed_threshold,
        "candidate_threshold_low": self.candidate_threshold_low,
        "candidate_threshold_high": self.candidate_threshold_high,
        "llm_enabled": self.llm_enabled,
        "groq_model": GROQ_MODEL,
        "cross_modal_merge_min": CROSS_MODAL_MERGE_MIN,
        "merge_composite_min": MERGE_COMPOSITE_MIN,
      },
      "total_processing_time_sec": round(total_time, 6),
      "stage_timings": final_timings,
      "created_at": datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }


## 10. Public API — `resolve_entities()`


In [10]:
def resolve_entities(
  observations_payload: Dict[str, Any],
  config: Optional[Dict[str, Any]] = None,
  human_constraints: Optional[HumanConstraints] = None,
  llm_enabled: bool = True,
) -> Dict[str, Any]:
  return EntityResolutionPipeline(config=config, human_constraints=human_constraints, llm_enabled=llm_enabled).run(observations_payload)


## 11a. Built-in test observation payloads


In [11]:
ATM_DEMO_INPUT = {
  "case_id": "CASE_ATM_004",
  "observations": [
    {"obs_id": "O1", "entity": "Person_97", "role": "suspect", "modality": "video", "location": "ATM booth entrance, exterior facing", "content": "Individual seen walking towards ATM kiosk (captured on footage).", "timestamp": "2024-01-15T10:01:20", "confidence": 0.906},
    {"obs_id": "O2", "entity": "Speaker_K", "role": "suspect", "modality": "audio", "location": "near ATM location, mobile network", "content": "I can see the booth from here.", "timestamp": "2024-01-15T10:01:12", "confidence": 0.613},
    {"obs_id": "O3", "entity": "sms_35", "role": "suspect", "modality": "text", "location": "remote — email server log", "content": "Heading there. Any activity?", "timestamp": "2024-01-15T10:00:55", "confidence": 0.683},
    {"obs_id": "O7", "entity": "Person_97", "role": "suspect", "modality": "video", "location": "ATM booth interior, card reader and keypad area", "content": "Suspect enters the ATM booth unaccompanied (captured on footage).", "timestamp": "2024-01-15T10:03:54", "confidence": 0.875},
    {"obs_id": "O8", "entity": "Speaker_K", "role": "suspect", "modality": "audio", "location": "ATM vicinity, intercepted channel", "content": "I'm in, it's clear.", "timestamp": "2024-01-15T10:03:56", "confidence": 0.778},
    {"obs_id": "O9", "entity": "sms_35", "role": "suspect", "modality": "text", "location": "local police station, complaint desk", "content": "Entered. Starting now.", "timestamp": "2024-01-15T10:04:06", "confidence": 0.730},
    {"obs_id": "O14", "entity": "Speaker_K", "role": "suspect", "modality": "audio", "location": "ATM vicinity, intercepted channel", "content": "Done, I'm coming out.", "timestamp": "2024-01-15T10:04:58", "confidence": 0.836},
    {"obs_id": "O15", "entity": "sms_35", "role": "suspect", "modality": "text", "location": "bank security office, incident registry", "content": "Exited. Don't wait for me.", "timestamp": "2024-01-15T10:04:57", "confidence": 0.785},
    {"obs_id": "O16", "entity": "Person_32", "role": "witness", "modality": "video", "location": "ATM lobby doorway, entry/exit point", "content": "Eyewitness notices a person leaving the ATM enclosure in a rush.", "timestamp": "2024-01-15T10:05:09", "confidence": 0.482},
    {"obs_id": "O17", "entity": "Person_32", "role": "witness", "modality": "video", "location": "ATM booth entrance, exterior facing", "content": "Bystander seen speaking with bank security personnel outside.", "timestamp": "2024-01-15T10:08:44", "confidence": 0.501},
  ],
}

IP_LOG_DEMO = {
  "case_id": "CASE_IP_CROSSMODAL_001",
  "observations": [
    {"obs_id": "T1", "entity": "Suspect A", "role": "suspect", "modality": "text", "location": "interview room transcript", "content": "Suspect A stated they accessed the server at 14:02.", "timestamp": "2024-03-01T14:05:00", "confidence": 0.82},
    {"obs_id": "L1", "entity": "192.168.4.22", "role": "suspect", "modality": "text", "location": "firewall log / dmz segment", "content": "SSH login accepted from 192.168.4.22 at 14:02:11.", "timestamp": "2024-03-01T14:02:11", "confidence": 0.91},
    {"obs_id": "V1", "entity": "individual_in_red_jacket", "role": "suspect", "modality": "video", "location": "server room hallway camera 3", "content": "Individual in red jacket badged into server room at 14:01:50.", "timestamp": "2024-03-01T14:01:50", "confidence": 0.88},
    {"obs_id": "W1", "entity": "Security Guard", "role": "witness", "modality": "audio", "location": "server room entrance", "content": "Guard reports unknown person in red jacket near rack 4.", "timestamp": "2024-03-01T14:03:00", "confidence": 0.74},
  ],
}


## 11b. Demo & test runner

Runs built-in **ATM** and **IP/cross-modal** scenarios. Optionally set `CASE_FILE_PATH` to a `*_obs_only.json` file.


In [12]:
import os, sys, json

# Optional: path to your observation JSON file
CASE_FILE_PATH = os.environ.get('FORENSYNTH_CASE_FILE')  # e.g. r'C:\\data\\CASE_ATM_004_obs_only.json'

cases = [('ATM multi-modal suspect', ATM_DEMO_INPUT), ('IP + transcript + video', IP_LOG_DEMO)]
if CASE_FILE_PATH and os.path.exists(CASE_FILE_PATH):
    with open(CASE_FILE_PATH, encoding='utf-8') as fh:
        cases = [('custom file', json.load(fh))]

LLM_ON = bool(os.environ.get('GROQ_API_KEY'))
print(f'LLM enabled: {LLM_ON}')

for label, payload in cases:
    print('\n' + '=' * 72)
    print(f'CASE: {label} ({payload["case_id"]})')
    result = resolve_entities(payload, llm_enabled=LLM_ON)
    print(f'status={result["status"]} entities={result["entity_count"]} conflicts={result["conflicts_detected"]}')
    for ent in result['canonical_entities']:
        print(f"  - {ent['entity_id']}: aliases={ent['aliases']} modalities={ent['modalities']}")
    if payload['case_id'] == 'CASE_ATM_004':
        assert result['entity_count'] <= 3
        suspect = next(e for e in result['canonical_entities'] if 'suspect' in e['roles'])
        assert len(suspect['aliases']) >= 2
    if payload['case_id'] == 'CASE_IP_CROSSMODAL_001':
        suspects = [e for e in result['canonical_entities'] if 'suspect' in e['roles']]
        assert len(suspects) == 1 and len(suspects[0]['aliases']) >= 2
print('\n[OK] Demo assertions passed.')



LLM enabled: True

CASE: ATM multi-modal suspect (CASE_ATM_004)


status=awaiting_human_validation entities=2 conflicts=1
  - entity_1: aliases=['Person_97', 'Speaker_K', 'sms_35'] modalities=['audio', 'text', 'video']
  - entity_2: aliases=['Person_32'] modalities=['video']

CASE: IP + transcript + video (CASE_IP_CROSSMODAL_001)
status=success entities=2 conflicts=0
  - entity_1: aliases=['192.168.4.22', 'Suspect A', 'individual_in_red_jacket'] modalities=['text', 'video']
  - entity_2: aliases=['Security Guard'] modalities=['audio']

[OK] Demo assertions passed.


## 12. Timeline Agent payload

Extracts the contract fields your **Timeline Agent** consumes.


In [13]:
timeline_agent_payload = {
    'case_id': result['case_id'],
    'canonical_entities': result['canonical_entities'],
    'clusters': result['clusters'],
    'conflicts_detected': result['conflicts_detected'],
}
print(json.dumps(timeline_agent_payload, indent=2))



{
  "case_id": "CASE_IP_CROSSMODAL_001",
  "canonical_entities": [
    {
      "entity_id": "entity_1",
      "aliases": [
        "192.168.4.22",
        "Suspect A",
        "individual_in_red_jacket"
      ],
      "primary_alias": "192.168.4.22",
      "total_mentions": 3,
      "confirmed_mentions": [],
      "candidate_mentions": [],
      "confidence_score": 0.87,
      "confirmed_edges": 0,
      "candidate_edges": 2,
      "modalities": [
        "text",
        "video"
      ],
      "locations": [
        "firewall log / dmz segment",
        "interview room transcript",
        "server room hallway camera 3"
      ],
      "roles": [
        "suspect"
      ],
      "sources": [
        "L1",
        "T1",
        "V1"
      ],
      "earliest_timestamp": "2024-03-01T14:01:50Z",
      "latest_timestamp": "2024-03-01T14:05:00Z",
      "time_span_seconds": 190
    },
    {
      "entity_id": "entity_2",
      "aliases": [
        "Security Guard"
      ],
      "primary_alias

## 13. Validate output contract


In [14]:
EXPECTED_TIMING_KEYS = {
    'stage_1_intake', 'stage_2_normalization', 'stage_3_blocking', 'stage_4_features',
    'stage_5_scoring', 'stage_6_classification', 'stage_7_graph_building', 'stage_8_clustering',
    'stage_9_attachment', 'stage_10_conflict_detection', 'stage_11_labeling', 'stage_12_packaging',
}
actual = set(result['stage_timings'].keys())
assert EXPECTED_TIMING_KEYS <= actual, f'missing keys: {EXPECTED_TIMING_KEYS - actual}'
print('[OK] All stage_timings keys present.')



[OK] All stage_timings keys present.
